In [14]:
import pandas as pd
import numpy as np
from collections import defaultdict

# ================================================
# CONFIG - Tweak these based on your floor reality
# ================================================
FILE_PATH = "D:/Tushar/Copy of Master Data _290102026 2 - Copy.xlsx"  # ← your exact file name

TARGET_COVERAGE_DAYS = 3.0          # Build up to 3 days buffer when we run a part
MAX_PARTS_PER_MACHINE = 3           # Your strict rule: max 2-3 parts / machine / day
MAX_HOURS_PER_DAY = 22.0
CHANGEOVER_MIN = 40                 # minutes
CHANGEOVER_HOURS = CHANGEOVER_MIN / 60

# RRS thresholds (adjust after first run)
RUNNER_DEMAND_THRESHOLD = 500       # Runner = daily high-volume (fixed machines)
REPEATER_DEMAND_THRESHOLD = 50      # Repeater = medium, we build 3-day lots

# Sheet name (from your previous notebooks)
SHEET_NAME = "Master Data "

# ================================================
# 1. LOAD & AGGREGATE CORRECTLY (exactly as you described)
# ================================================
master = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

# Clean columns
master["Daily Plan"] = pd.to_numeric(master["Daily Plan"], errors="coerce").fillna(0)
master["Sub Count"] = pd.to_numeric(master["Sub Count"], errors="coerce").fillna(0)
master["Inventory_25"] = pd.to_numeric(master["Inventory_25"], errors="coerce").fillna(0)
master["Minimum Quantity"] = pd.to_numeric(master["Minimum Quantity"], errors="coerce").fillna(0)
master["Cycle Time"] = pd.to_numeric(master["Cycle Time"], errors="coerce").fillna(0)   # seconds assumed

# Aggregate per unique Child Part
agg = []
for child, group in master.groupby("Child Part"):
    daily_needed = (group["Daily Plan"] * group["Sub Count"]).sum()
    min_qty = group["Minimum Quantity"].iloc[0]          # add only once
    inventory = group["Inventory_25"].iloc[0]
    cycle_time = group["Cycle Time"].iloc[0]              # assume same for the part
    color = group["Color"].iloc[0] if "Color" in group.columns else "Unknown"
    machines_raw = ",".join(group["Vertical Machines"].astype(str).unique())
    
    net_required = daily_needed + min_qty - inventory
    net_required = max(0, net_required)
    
    agg.append({
        "Child Part": child,
        "Daily_Demand": daily_needed,          # today's true demand
        "Net_Required": net_required,          # daily + one-time buffer - inv
        "Cycle_Time_sec": cycle_time,
        "Color": color,
        "Vertical_Machines": machines_raw,
        "Current_Inventory": inventory
    })

df = pd.DataFrame(agg)

# Classify Runner / Repeater / Stranger
df["Category"] = "Stranger"
df.loc[df["Daily_Demand"] >= RUNNER_DEMAND_THRESHOLD, "Category"] = "Runner"
df.loc[(df["Daily_Demand"] >= REPEATER_DEMAND_THRESHOLD) & 
       (df["Daily_Demand"] < RUNNER_DEMAND_THRESHOLD), "Category"] = "Repeater"

print("Classification:")
print(df["Category"].value_counts())
print("\nSample (highest net required):")
print(df.sort_values("Net_Required", ascending=False).head(10)[["Child Part","Daily_Demand","Net_Required","Category"]])

# We only schedule Repeater + Stranger (Runners are on fixed machines)
to_schedule = df[df["Category"].isin(["Repeater", "Stranger"])].copy()

# ================================================
# 2. PREPARE ELIGIBLE MACHINES PER PART
# ================================================
def normalize_machines(s):
    if pd.isna(s): return []
    return [m.strip().upper() for m in str(s).split(",") if m.strip()]

to_schedule["Eligible_Machines"] = to_schedule["Vertical_Machines"].apply(normalize_machines)

# ================================================
# 3. TWO-PHASE SCHEDULER (Daily first → Buffer second)
# ================================================
machines = ["MP-01", "MP-05", "MP-10", "MP-11", "MP-17"]
machine_load = {m: 0.0 for m in machines}
machine_sequence = {m: [] for m in machines}   # list of parts in order (for color grouping)

schedule = []

# PHASE 1: Guarantee TODAY'S DAILY DEMAND (split if needed)
print("\n=== PHASE 1: Covering today's daily demand ===")
daily_df = to_schedule.copy()
daily_df["Remaining_Daily"] = daily_df["Daily_Demand"]

for _, part in daily_df.iterrows():
    if part["Remaining_Daily"] <= 0:
        continue
    
    prod_time_per_piece = part["Cycle_Time_sec"] / 3600
    qty_left = part["Remaining_Daily"]
    eligible = part["Eligible_Machines"]
    
    for m in eligible:
        if m not in machines: continue
        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS: continue
            
        # Setup only if first part or different color
        setup_h = CHANGEOVER_HOURS if not machine_sequence[m] or machine_sequence[m][-1]["Color"] != part["Color"] else 0
        free -= setup_h
        if free <= 0: continue
        
        max_qty_fit = free / prod_time_per_piece
        assign_qty = min(qty_left, max_qty_fit)
        if assign_qty <= 0: continue
        
        assign_hours = assign_qty * prod_time_per_piece
        
        # Record
        machine_load[m] += assign_hours + setup_h
        machine_sequence[m].append({"Child Part": part["Child Part"], "Color": part["Color"], "Qty": assign_qty, "Hours": assign_hours, "Setup": setup_h})
        
        schedule.append({
            "Machine": m,
            "Child Part": part["Child Part"],
            "Qty": round(assign_qty, 0),
            "Type": "Daily",
            "Hours": round(assign_hours + setup_h, 2),
            "Color": part["Color"]
        })
        
        qty_left -= assign_qty
        if qty_left <= 0: break

# PHASE 2: Add buffer (big lots) to Repeaters + urgent Strangers, max 3 parts/machine total
print("\n=== PHASE 2: Adding 3-day buffer where capacity allows ===")
buffer_df = to_schedule[to_schedule["Net_Required"] > to_schedule["Daily_Demand"]].copy()  # only parts that still need buffer
buffer_df["Buffer_Qty"] = buffer_df["Net_Required"] - buffer_df["Daily_Demand"]
buffer_df = buffer_df.sort_values("Buffer_Qty", ascending=False)

for _, part in buffer_df.iterrows():
    qty_left = part["Buffer_Qty"]
    if qty_left <= 0: continue
    
    prod_time_per_piece = part["Cycle_Time_sec"] / 3600
    eligible = part["Eligible_Machines"]
    lot_target = part["Daily_Demand"] * TARGET_COVERAGE_DAYS   # big lot
    
    # Try to assign big lot (or portion) without exceeding 3 parts/machine
    for m in sorted(eligible, key=lambda x: machine_load.get(x, 0)):
        if m not in machines: continue
        if len(machine_sequence[m]) >= MAX_PARTS_PER_MACHINE: continue
            
        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS: continue
            
        setup_h = CHANGEOVER_HOURS if not machine_sequence[m] or machine_sequence[m][-1]["Color"] != part["Color"] else 0
        free -= setup_h
        if free <= 0: continue
        
        max_qty_fit = free / prod_time_per_piece
        assign_qty = min(qty_left, lot_target, max_qty_fit)
        if assign_qty < 50: continue   # ignore tiny buffer adds
        
        assign_hours = assign_qty * prod_time_per_piece
        
        machine_load[m] += assign_hours + setup_h
        machine_sequence[m].append({"Child Part": part["Child Part"], "Color": part["Color"], "Qty": assign_qty, "Hours": assign_hours, "Setup": setup_h})
        
        schedule.append({
            "Machine": m,
            "Child Part": part["Child Part"],
            "Qty": round(assign_qty, 0),
            "Type": "Buffer",
            "Hours": round(assign_hours + setup_h, 2),
            "Color": part["Color"]
        })
        
        qty_left -= assign_qty
        if qty_left <= 0: break

# ================================================
# 4. FINAL OUTPUT
# ================================================
print("\n" + "="*90)
print("FINAL DAILY PLAN (Daily demand guaranteed + smart buffer)")
print("="*90)

total_hours = 0
total_changeovers = 0

for m in machines:
    parts = machine_sequence.get(m, [])
    hours = machine_load.get(m, 0)
    total_hours += hours
    
    print(f"\n🛠 {m}   {hours:6.1f} / {MAX_HOURS_PER_DAY:.1f} h   ({hours/MAX_HOURS_PER_DAY*100:5.1f}%)")
    if not parts:
        print("   No Repeater/Stranger assigned")
        continue
    
    changeovers = sum(1 for i in range(1, len(parts)) if parts[i]["Color"] != parts[i-1]["Color"])
    total_changeovers += changeovers
    
    for p in parts:
        setup_min = p["Setup"] * 60
        print(f"   {p['Child Part']:20}  {p['Qty']:>6} pcs   {p['Hours']:>5.1f}h  (setup {setup_min:>3.0f} min)   {p['Type']}")

print("\n" + "-"*90)
print(f"Total hours used     : {total_hours:.1f} h across all machines")
print(f"Estimated changeovers: {total_changeovers}")
print(f"Parts scheduled      : {len(schedule)}")
print(f"Daily demand covered : 100% (by design)")
print("-"*90)

# Save
pd.DataFrame(schedule).to_excel("daily_plan_with_buffer.xlsx", index=False)
print("Saved → daily_plan_with_buffer.xlsx")

Classification:
Category
Stranger    14565
Repeater     1611
Runner       1395
Name: count, dtype: int64

Sample (highest net required):
          Child Part  Daily_Demand   Net_Required Category
4730   E-RES-00025Z0        1336.0  301163.838710   Runner
11672  S21008-023A0Y       10604.0  226784.096774   Runner
4727   E-RES-00022Z0        1336.0  205002.129032   Runner
12602  S22127-007A0X        6942.0  198073.354839   Runner
16417  S32197-009A0Z        2808.0  162819.967742   Runner
13713  S31125-008A0Z        4208.0  160409.967742   Runner
16686  S33056-010A0Z        9632.0  135470.580645   Runner
16869  S33081-008A0Z       16718.0  131920.032258   Runner
11676  S21008-028A0Y        6870.0  125500.451613   Runner
16507  S33038-021A0Z        9533.0  114663.451613   Runner

=== PHASE 1: Covering today's daily demand ===

=== PHASE 2: Adding 3-day buffer where capacity allows ===

FINAL DAILY PLAN (Daily demand guaranteed + smart buffer)

🛠 MP-01      0.0 / 22.0 h   (  0.0%)
   No Rep